# 03 — TabNet Classification on Fused QR-Code Features

This notebook trains and evaluates the TabNet branch of:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

It consumes the preprocessed fused feature partitions produced by:

- `01_dataset_preparation.ipynb`
- `02_feature_extraction_and_fusion.ipynb`

## Outputs

- trained TabNet model;
- validation and test metrics;
- predictions and classification reports;
- confusion matrices;
- ROC and precision–recall curves;
- feature-importance tables and figures;
- structural-versus-statistical importance summary.

## 1. Environment and Reproducibility

Install `pytorch-tabnet` before running this notebook when it is not already available:

```bash
pip install pytorch-tabnet
```

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("Random seed    :", SEED)

## 2. Repository Paths

This notebook supports execution from either the repository root or the `Notebooks/` directory.

In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FUSED_DIR = PROCESSED_DATA_DIR / "fused"

MODELS_DIR = PROJECT_ROOT / "Models"
TABNET_MODEL_DIR = MODELS_DIR / "tabnet"

RESULTS_DIR = PROJECT_ROOT / "Results"
TABNET_RESULTS_DIR = RESULTS_DIR / "tabnet"
FIGURES_DIR = TABNET_RESULTS_DIR / "figures"
TABLES_DIR = TABNET_RESULTS_DIR / "tables"
METRICS_DIR = TABNET_RESULTS_DIR / "metrics"

for directory in [
    TABNET_MODEL_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    METRICS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_FUSED_CSV = FUSED_DIR / "train_fused_preprocessed.csv"
VAL_FUSED_CSV = FUSED_DIR / "val_fused_preprocessed.csv"
TEST_FUSED_CSV = FUSED_DIR / "test_fused_preprocessed.csv"

print("Project root :", PROJECT_ROOT)
print("TabNet model :", TABNET_MODEL_DIR)
print("TabNet results:", TABNET_RESULTS_DIR)

## 3. Load and Validate Fused Feature Partitions

In [ ]:
for path in [TRAIN_FUSED_CSV, VAL_FUSED_CSV, TEST_FUSED_CSV]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Run Notebook 02 first."
        )

train_df = pd.read_csv(TRAIN_FUSED_CSV).copy()
val_df = pd.read_csv(VAL_FUSED_CSV).copy()
test_df = pd.read_csv(TEST_FUSED_CSV).copy()

print("Train shape:", train_df.shape)
print("Val shape  :", val_df.shape)
print("Test shape :", test_df.shape)

meta_cols = ["image", "label", "label_id"]

for name, dataframe in {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}.items():
    missing = [column for column in meta_cols if column not in dataframe.columns]
    if missing:
        raise ValueError(f"{name} split is missing columns: {missing}")

    dataframe["image"] = dataframe["image"].astype(str).str.strip()
    dataframe["label"] = dataframe["label"].astype(str).str.strip().str.lower()
    dataframe["label_id"] = pd.to_numeric(
        dataframe["label_id"], errors="coerce"
    )

    if dataframe["label_id"].isna().any():
        raise ValueError(f"{name} split contains invalid label IDs.")

    dataframe["label_id"] = dataframe["label_id"].astype(int)

feature_cols = [
    column for column in train_df.columns if column not in meta_cols
]

if not feature_cols:
    raise ValueError("No fused feature columns were found.")

missing_in_val = [
    column for column in feature_cols if column not in val_df.columns
]
missing_in_test = [
    column for column in feature_cols if column not in test_df.columns
]

if missing_in_val:
    raise ValueError(
        f"Validation split is missing feature columns: {missing_in_val}"
    )

if missing_in_test:
    raise ValueError(
        f"Test split is missing feature columns: {missing_in_test}"
    )

train_df = train_df[meta_cols + feature_cols].copy()
val_df = val_df[meta_cols + feature_cols].copy()
test_df = test_df[meta_cols + feature_cols].copy()

## 4. Prepare Model Inputs

In [ ]:
X_train = train_df[feature_cols].astype(np.float32).to_numpy()
y_train = train_df["label_id"].astype(int).to_numpy()

X_val = val_df[feature_cols].astype(np.float32).to_numpy()
y_val = val_df["label_id"].astype(int).to_numpy()

X_test = test_df[feature_cols].astype(np.float32).to_numpy()
y_test = test_df["label_id"].astype(int).to_numpy()

for name, matrix in {
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
}.items():
    if np.isnan(matrix).any():
        raise ValueError(f"{name} contains NaN values.")
    if np.isinf(matrix).any():
        raise ValueError(f"{name} contains infinite values.")

print("Feature count:", len(feature_cols))
print("Train classes:", np.bincount(y_train))
print("Val classes  :", np.bincount(y_val))
print("Test classes :", np.bincount(y_test))

data_config = {
    "seed": SEED,
    "feature_columns": feature_cols,
    "feature_count": len(feature_cols),
    "target_column": "label_id",
    "train_shape": list(X_train.shape),
    "validation_shape": list(X_val.shape),
    "test_shape": list(X_test.shape),
}

DATA_CONFIG_PATH = TABNET_MODEL_DIR / "tabnet_data_config.json"

with DATA_CONFIG_PATH.open("w", encoding="utf-8") as file:
    json.dump(data_config, file, indent=2)

print("Saved data configuration:", DATA_CONFIG_PATH)

## 5. Build and Train TabNet

The following configuration is retained from the original experiment:

- decision width (`n_d`): 48;
- attention width (`n_a`): 48;
- decision steps: 5;
- optimizer: Adam;
- learning rate: 0.008;
- mask type: Entmax;
- scheduler: StepLR;
- maximum epochs: 150;
- patience: 25;
- batch size: 2048;
- virtual batch size: 256.

In [ ]:
tabnet_model = TabNetClassifier(
    n_d=48,
    n_a=48,
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params={"lr": 8e-3},
    mask_type="entmax",
    scheduler_params={"step_size": 15, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=SEED,
    verbose=1,
)

tabnet_model.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_name=["train", "val"],
    eval_metric=["auc"],
    max_epochs=150,
    patience=25,
    batch_size=2048,
    virtual_batch_size=256,
    num_workers=0,
    drop_last=False,
)

MODEL_BASE = TABNET_MODEL_DIR / "tabnet_fused_model"
tabnet_model.save_model(str(MODEL_BASE))

print("Saved model:", str(MODEL_BASE) + ".zip")

## 6. Validation Evaluation

In [ ]:
val_pred = tabnet_model.predict(X_val)
val_proba = tabnet_model.predict_proba(X_val)[:, 1]

validation_metrics = pd.DataFrame(
    [
        {
            "accuracy": accuracy_score(y_val, val_pred),
            "precision": precision_score(
                y_val, val_pred, zero_division=0
            ),
            "recall": recall_score(
                y_val, val_pred, zero_division=0
            ),
            "f1_score": f1_score(
                y_val, val_pred, zero_division=0
            ),
            "roc_auc": roc_auc_score(y_val, val_proba),
        }
    ]
)

validation_metrics.to_csv(
    METRICS_DIR / "tabnet_validation_metrics.csv",
    index=False,
)

validation_predictions = val_df[meta_cols].copy()
validation_predictions["y_true"] = y_val
validation_predictions["y_pred"] = val_pred
validation_predictions["y_proba"] = val_proba
validation_predictions.to_csv(
    TABLES_DIR / "tabnet_validation_predictions.csv",
    index=False,
)

validation_report = classification_report(
    y_val,
    val_pred,
    target_names=["benign", "malicious"],
    digits=4,
)

with (METRICS_DIR / "tabnet_validation_report.txt").open(
    "w", encoding="utf-8"
) as file:
    file.write(validation_report)

print(validation_metrics)
print()
print(validation_report)

## 7. Test Evaluation and Inference Time

In [ ]:
from time import perf_counter

start_time = perf_counter()
test_proba_full = tabnet_model.predict_proba(X_test)
inference_seconds = perf_counter() - start_time

test_proba = test_proba_full[:, 1]
test_pred = np.argmax(test_proba_full, axis=1)

test_metrics = pd.DataFrame(
    [
        {
            "accuracy": accuracy_score(y_test, test_pred),
            "precision": precision_score(
                y_test, test_pred, zero_division=0
            ),
            "recall": recall_score(
                y_test, test_pred, zero_division=0
            ),
            "f1_score": f1_score(
                y_test, test_pred, zero_division=0
            ),
            "roc_auc": roc_auc_score(y_test, test_proba),
            "inference_seconds": inference_seconds,
            "milliseconds_per_sample": (
                inference_seconds / len(X_test) * 1000
            ),
        }
    ]
)

test_metrics.to_csv(
    METRICS_DIR / "tabnet_test_metrics.csv",
    index=False,
)

test_predictions = test_df[meta_cols].copy()
test_predictions["y_true"] = y_test
test_predictions["y_pred"] = test_pred
test_predictions["y_proba"] = test_proba
test_predictions.to_csv(
    TABLES_DIR / "tabnet_test_predictions.csv",
    index=False,
)

test_report = classification_report(
    y_test,
    test_pred,
    target_names=["benign", "malicious"],
    digits=4,
)

with (METRICS_DIR / "tabnet_test_report.txt").open(
    "w", encoding="utf-8"
) as file:
    file.write(test_report)

print(test_metrics)
print()
print(test_report)

## 8. Confusion Matrices

In [ ]:
cm_val = confusion_matrix(y_val, val_pred)
cm_test = confusion_matrix(y_test, test_pred)

pd.DataFrame(
    cm_val,
    index=["true_benign", "true_malicious"],
    columns=["pred_benign", "pred_malicious"],
).to_csv(TABLES_DIR / "tabnet_validation_confusion_matrix.csv")

pd.DataFrame(
    cm_test,
    index=["true_benign", "true_malicious"],
    columns=["pred_benign", "pred_malicious"],
).to_csv(TABLES_DIR / "tabnet_test_confusion_matrix.csv")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=["Benign", "Malicious"],
).plot(ax=axes[0], colorbar=False, values_format="d")
axes[0].set_title("TabNet Validation Confusion Matrix")

ConfusionMatrixDisplay(
    confusion_matrix=cm_test,
    display_labels=["Benign", "Malicious"],
).plot(ax=axes[1], colorbar=False, values_format="d")
axes[1].set_title("TabNet Test Confusion Matrix")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "tabnet_validation_test_confusion_matrices.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 9. ROC Curves

In [ ]:
fpr_val, tpr_val, _ = roc_curve(y_val, val_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, test_proba)

roc_auc_val = roc_auc_score(y_val, val_proba)
roc_auc_test = roc_auc_score(y_test, test_proba)

roc_points = pd.concat(
    [
        pd.DataFrame(
            {
                "split": "validation",
                "false_positive_rate": fpr_val,
                "true_positive_rate": tpr_val,
            }
        ),
        pd.DataFrame(
            {
                "split": "test",
                "false_positive_rate": fpr_test,
                "true_positive_rate": tpr_test,
            }
        ),
    ],
    ignore_index=True,
)

roc_points.to_csv(
    TABLES_DIR / "tabnet_roc_curve_points.csv",
    index=False,
)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_val,
    tpr_val,
    label=f"Validation ROC (AUC = {roc_auc_val:.4f})",
    linewidth=2,
)
plt.plot(
    fpr_test,
    tpr_test,
    linestyle="--",
    label=f"Test ROC (AUC = {roc_auc_test:.4f})",
    linewidth=2,
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle=":",
    linewidth=1.5,
    label="Random classifier",
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — TabNet Fused Features")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "tabnet_roc_curve_validation_test.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 10. Precision–Recall Curves

In [ ]:
precision_val, recall_val, _ = precision_recall_curve(
    y_val, val_proba
)
precision_test, recall_test, _ = precision_recall_curve(
    y_test, test_proba
)

ap_val = average_precision_score(y_val, val_proba)
ap_test = average_precision_score(y_test, test_proba)

pr_points = pd.concat(
    [
        pd.DataFrame(
            {
                "split": "validation",
                "recall": recall_val,
                "precision": precision_val,
            }
        ),
        pd.DataFrame(
            {
                "split": "test",
                "recall": recall_test,
                "precision": precision_test,
            }
        ),
    ],
    ignore_index=True,
)

pr_points.to_csv(
    TABLES_DIR / "tabnet_precision_recall_curve_points.csv",
    index=False,
)

baseline = float(np.mean(y_test))

plt.figure(figsize=(8, 6))
plt.plot(
    recall_val,
    precision_val,
    label=f"Validation PR (AP = {ap_val:.4f})",
)
plt.plot(
    recall_test,
    precision_test,
    linestyle="--",
    label=f"Test PR (AP = {ap_test:.4f})",
)
plt.hlines(
    baseline,
    xmin=0,
    xmax=1,
    linestyles=":",
    label=f"Baseline ({baseline:.4f})",
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve — TabNet Fused Features")
plt.legend(loc="lower left")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "tabnet_precision_recall_validation_test.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Feature Importance

In [ ]:
feature_importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": tabnet_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

feature_importance_df.to_csv(
    TABLES_DIR / "tabnet_feature_importance.csv",
    index=False,
)

top_n = min(30, len(feature_importance_df))
plot_df = feature_importance_df.head(top_n)

plt.figure(figsize=(11, 6))
plt.bar(plot_df["feature"], plot_df["importance"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Importance")
plt.title(f"TabNet Feature Importance — Top {top_n}")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "tabnet_feature_importance_top30.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

display(feature_importance_df.head(30))

## 12. Structural and Statistical Importance Summary

In [ ]:
importance_summary = pd.DataFrame(
    {
        "group": ["structural", "statistical"],
        "total_importance": [
            feature_importance_df.loc[
                feature_importance_df["feature"].str.startswith(
                    "struct_"
                ),
                "importance",
            ].sum(),
            feature_importance_df.loc[
                feature_importance_df["feature"].str.startswith(
                    "stat_"
                ),
                "importance",
            ].sum(),
        ],
    }
)

importance_summary.to_csv(
    TABLES_DIR / "tabnet_feature_group_importance.csv",
    index=False,
)

plt.figure(figsize=(6, 4))
plt.bar(
    importance_summary["group"],
    importance_summary["total_importance"],
)
plt.ylabel("Total Importance")
plt.title("TabNet Feature-Group Importance")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "tabnet_feature_group_importance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

display(importance_summary)

## 13. Final Result Summary

In [ ]:
final_summary = pd.concat(
    [
        validation_metrics.assign(split="validation"),
        test_metrics.assign(split="test"),
    ],
    ignore_index=True,
)

ordered_columns = [
    "split",
    "accuracy",
    "precision",
    "recall",
    "f1_score",
    "roc_auc",
]

remaining_columns = [
    column for column in final_summary.columns
    if column not in ordered_columns
]

final_summary = final_summary[
    ordered_columns + remaining_columns
]

final_summary.to_csv(
    METRICS_DIR / "tabnet_final_summary.csv",
    index=False,
)

print("=" * 68)
print("TABNET TRAINING AND EVALUATION COMPLETED")
print("=" * 68)
display(final_summary)
print("Model :", str(MODEL_BASE) + ".zip")
print("Results:", TABNET_RESULTS_DIR)
print("=" * 68)